# The whole thing, assembled — the hands-on half

The practical companion to **`e2e_slides.html`**. Every earlier session solved one problem. This
one connects them, and then does the thing that none of them can do alone:

> **take a single MLflow run id, and from it recover the exact code and the exact data, re-run the
> pipeline, and get the same metric back.**

That is what "reproducible" means operationally, and it is the point of the entire course.

| Part | From session | What you do |
|---|---|---|
| 1 · Version the data | 6 · DVC | `dvc add`, a remote, a committed pointer |
| 2 · A pipeline that cannot leak | 1 · sklearn | `ColumnTransformer` inside `dvc.yaml` stages |
| 3 · Record the run | 7 · MLflow | params, metrics, model — **plus two tags** |
| 4 · Serve it | 3 · FastAPI | load from the registry, validate, `/metrics` |
| 5 · Ship it | 4 · Docker | a real image, a real container answering |
| 6 · **Prove it** | all of them | run id → commit + data hash → same metric |
| 7 · What is still missing | — | the honest list |

### Eight words, before anything else

New to MLOps? These come up constantly below, so here they are in plain language:

| Word | What it means here |
|---|---|
| **run** | one execution of the training script. MLflow gives each one an **id** — a random string — and stores what it used and what it scored. |
| **commit** | git's snapshot of your code, named by a hash. "Which code trained this?" is answered by a commit. |
| **clean tree** | git-speak for "no unsaved edits". With uncommitted edits, the commit hash no longer describes what actually ran. |
| **fingerprint** | a short string computed from a file's bytes. Same bytes → same fingerprint. It lets you say *which* data without carrying the data. |
| **artifact** | any file a run produces and you want to keep — usually the model. |
| **provenance** | the paper trail: which data, which code, which run produced the thing in front of you. |
| **stage** | one step of a pipeline — "run `train.py`" — written in `dvc.yaml` with what it reads and writes. |
| **remote** | where the data files actually live: a folder, an S3 bucket, a shared drive. Git stores the pointer; the remote stores the bytes. |
| **mount** | making a folder on your machine visible inside a container, instead of copying it into the image. |

Everything happens in a throwaway `e2e_demo/` folder; the last cell removes it and its image.

## Step 0.1 · Install and check the tools

In [1]:
!pip install -q dvc mlflow scikit-learn pandas fastapi uvicorn 'pydantic>=2' prometheus-client requests joblib

### Are the three tools actually there?

This session needs **DVC** (versions the data), **MLflow** (records the runs) and **Docker**
(packages the service). The cell below asks each one for its version. If Docker says
`NOT AVAILABLE`, everything except Part 5 still works.

In [2]:
import subprocess, sys
for cmd in (["dvc", "--version"], ["mlflow", "--version"], ["docker", "--version"]):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"  {cmd[0]:8}", (r.stdout or r.stderr).strip().splitlines()[0] if r.returncode == 0 else "NOT AVAILABLE")

  dvc      3.59.0
  mlflow   mlflow, version 3.1.4
  docker   Docker version 29.6.1, build 8900f1d


## Step 0.2 · A sandbox, and a git repo to hang everything on

In [3]:
import os, shutil, pathlib

BASE = pathlib.Path.cwd()          # the folder this notebook lives in
PROJ = BASE / "e2e_demo"           # a throwaway sandbox, deleted by the last cell

if PROJ.exists():
    shutil.rmtree(PROJ)            # re-running this notebook is always safe
PROJ.mkdir(parents=True)
os.chdir(PROJ)
print("working inside:", os.getcwd())

working inside: /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/13-end-to-end/e2e_demo


Now a **git repository**, because the whole session depends on being able to say *which code*.
`dvc init` adds DVC's bookkeeping alongside git's. Both are set up inside the sandbox, so your
own repositories are untouched.

In [4]:
!git init -q -b main
!git config user.email "e2e@qafza.local" && git config user.name "E2E Demo"
open(".gitignore", "w").write("__pycache__/\nmlruns/\nmlflow.db\n*.pyc\n")
!git add .gitignore && git commit -q -m "chore: init"
!dvc init -q && dvc config core.analytics false
!git add .dvc .dvcignore && git commit -q -m "chore: dvc init"
!git log --oneline

b9b97f8 (HEAD -> main) chore: dvc init
3cee913 chore: init


Four empty folders. `%%writefile` further down will not create a folder that does not exist —
it fails with `No such file or directory` — so they are made first.

In [5]:

import os
# %%writefile does NOT create parent directories -- make them before writing into them.
for d in ("src", "app", "data", "models"):
    os.makedirs(d, exist_ok=True)
open("app/__init__.py", "w").close()
print("directories ready:", sorted(os.listdir(".")))


directories ready: ['.dvc', '.dvcignore', '.git', '.gitignore', 'app', 'data', 'models', 'src']


---
# Part 1 — Version the data   ·   session 6

Nothing downstream is reproducible unless the input is identifiable. So this comes first.

In [6]:
import numpy as np, pandas as pd, os

os.makedirs("data", exist_ok=True)
rng = np.random.default_rng(0)
m = 5000
df = pd.DataFrame({
    "customer_id":    rng.integers(1, 1200, m),        # repeats -> a group, see session 1
    "tenure_months":  rng.integers(1, 72, m),
    "monthly_charge": rng.normal(65, 20, m).round(2),
    "support_calls":  rng.poisson(1.2, m),
    "plan":           rng.choice(["basic", "plus", "pro"], m, p=[.5, .3, .2]),
})
df.loc[rng.random(m) < .06, "monthly_charge"] = np.nan
s = (-0.04*df.tenure_months + 0.03*df.monthly_charge.fillna(65) + 0.55*df.support_calls
     + rng.normal(0, .8, m))
df["churn"] = (s > s.mean()).astype(int)
df.to_csv("data/raw.csv", index=False)
print(f"{len(df)} rows, {df.isna().sum().sum()} missing, churn rate {df.churn.mean():.3f}")

5000 rows, 317 missing, churn rate 0.498


/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


`dvc add` computes a **fingerprint** of the CSV, copies the file into DVC's cache, and writes a
small pointer file `data/raw.csv.dvc`. That pointer is what git stores. `dvc push` copies the
bytes to the **remote** — here just another folder, standing in for S3.

In [7]:
import subprocess
REMOTE = str(BASE / "e2e_dvc_remote")
os.makedirs(REMOTE, exist_ok=True)

!dvc add data/raw.csv -q
!dvc remote add -d storage "{REMOTE}" -f
!dvc push -q
!git add data/raw.csv.dvc data/.gitignore .dvc/config && git commit -q -m "data: raw.csv v1"

import yaml
DATA_MD5 = yaml.safe_load(open("data/raw.csv.dvc"))["outs"][0]["md5"]
print("data fingerprint:", DATA_MD5)
print("git carries:", os.path.getsize("data/raw.csv.dvc"), "bytes instead of",
      os.path.getsize("data/raw.csv"))

⠋ Checking graph
Setting 'storage' as a default remote.
data fingerprint: 977ddb47130abcf45cf9b16e2ccc74f7
git carries: 89 bytes instead of 109556


---
# Part 2 — A pipeline that cannot leak   ·   sessions 1 and 6

Two ideas combined: every fitted step lives inside a `Pipeline` (so preprocessing cannot see the
test fold), and the whole thing is a `dvc.yaml` stage (so the recipe is versioned with the code).

Note the splitter: `customer_id` repeats, so this uses `GroupKFold`. Session 1 measured what
ignoring that costs.

In [8]:
%%writefile params.yaml
test_size: 0.25
random_state: 42
n_estimators: 200
max_depth: 8

Writing params.yaml


This is the training script. Read it once: it is the only place in the session where a model is
fitted, and every idea from earlier sessions shows up in it — the `Pipeline` from session 1, the
MLflow tags from session 7, and the fingerprint from session 6.

In [9]:
%%writefile src/train.py
"""Trains the model, records the run, and writes metrics. One script, called by dvc repro."""
import json
import subprocess
from pathlib import Path

import joblib
import mlflow
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, cross_val_score, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM = ["tenure_months", "monthly_charge", "support_calls"]
CAT = ["plan"]
FEATURES = NUM + CAT

p = yaml.safe_load(open("params.yaml"))
df = pd.read_csv("data/raw.csv")

# session 1: every fitted step inside the Pipeline
model = Pipeline([
    ("pre", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])),
    ("clf", RandomForestClassifier(n_estimators=p["n_estimators"],
                                   max_depth=p["max_depth"],
                                   random_state=p["random_state"])),
])

# session 1: customers repeat, so hold out whole customers -- not random rows
splitter = GroupShuffleSplit(n_splits=1, test_size=p["test_size"],
                            random_state=p["random_state"])
train_idx, test_idx = next(splitter.split(df, groups=df.customer_id))
train, test = df.iloc[train_idx], df.iloc[test_idx]

model.fit(train[FEATURES], train.churn)
proba = model.predict_proba(test[FEATURES])[:, 1]
metrics = {
    "roc_auc":  round(float(roc_auc_score(test.churn, proba)), 4),
    "accuracy": round(float(accuracy_score(test.churn, proba >= .5)), 4),
    "f1":       round(float(f1_score(test.churn, proba >= .5)), 4),
    "n_train":  int(len(train)),
    "n_test":   int(len(test)),
}

# the two tags that make this run reproducible (session 6 + session 7)
data_md5 = yaml.safe_load(open("data/raw.csv.dvc"))["outs"][0]["md5"]
commit = subprocess.run(["git", "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
# A commit hash only describes the run if the tree matched it. Uncommitted edits make
# git_commit a lie, so record whether that was the case instead of hoping it was not.
#
# One catch, found the hard way: `dvc repro` DELETES a stage's own outputs before
# rerunning it, so while this script runs, `git status` reports `D metrics.json` --
# dvc at work, not somebody's uncommitted edit. A provenance check has to know what
# its own tooling touches, so the stage's outputs are excluded here.
OUTS = {"metrics.json", "run_id.txt", "models/model.joblib"}
status = subprocess.run(["git", "status", "--porcelain"],
                        capture_output=True, text=True).stdout.splitlines()
changed = [line[3:] for line in status if line[3:] not in OUTS]
dirty = bool(changed)
if dirty:
    print("WARNING: uncommitted changes -- git_commit does not describe this run:", changed)

mlflow.set_tracking_uri(f"sqlite:///{Path.cwd()}/mlflow.db")
mlflow.set_experiment("e2e-churn")
with mlflow.start_run(run_name="pipeline") as run:
    mlflow.log_params(p)
    mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, float)})
    mlflow.set_tag("dvc_data_md5", data_md5)      # exactly which data
    mlflow.set_tag("git_commit", commit)          # exactly which code
    mlflow.set_tag("splitter", "GroupShuffleSplit(customer_id)")
    mlflow.set_tag("git_dirty", str(dirty).lower())   # trust the run only if this is false
    Path("run_id.txt").write_text(run.info.run_id)

Path("models").mkdir(exist_ok=True)
joblib.dump({"pipeline": model, "features": FEATURES, "version": "1.0.0",
             "metrics": metrics, "run_id": Path("run_id.txt").read_text()},
            "models/model.joblib")
json.dump(metrics, open("metrics.json", "w"), indent=2)
print("metrics:", metrics)

Writing src/train.py


`dvc.yaml` describes the pipeline: the command to run, what it **reads** (`deps`), and what it
**writes** (`outs`). With that written down, `dvc repro` can work out whether anything needs to
run — and nobody has to remember the order.

In [10]:
%%writefile dvc.yaml
stages:
  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/raw.csv
    params:
      - test_size
      - random_state
      - n_estimators
      - max_depth
    outs:
      - models/model.joblib
      - run_id.txt
    metrics:
      - metrics.json:
          cache: false

Writing dvc.yaml


**Commit first, then run.** The training script tags each run with `git rev-parse HEAD` — the
current commit. If you run the pipeline *before* committing it, that tag points at a commit
that does not contain `dvc.yaml`, and Part 6 cannot rebuild anything from it. CI gets this
right for free, because a runner can only ever execute a commit that already exists.

In [11]:
# Commit the pipeline BEFORE running it. The run tags itself with `git rev-parse HEAD`,
# so if the commit comes afterwards the tag points at a tree without dvc.yaml in it and
# Part 6 cannot reproduce anything. CI gets this right for free: it always runs a commit.
!git add -A && git commit -q -m "feat: training pipeline"
!dvc repro


'data/raw.csv.dvc' didn't change, skipping
Running stage 'train':
> python src/train.py
/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2026/08/24 21:29:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/24 21:29:42 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgra

`dvc metrics show` reads `metrics.json` and prints it as a table. Then the outputs of the run
(the lock file and the metrics) get committed too.

In [12]:
!dvc metrics show
!git add -A && git commit -q -m "chore: lock and metrics"   # the run's outputs
PIPELINE_COMMIT = subprocess.run(["git", "log", "--format=%H", "-1", "--skip=1"],
                                 capture_output=True, text=True).stdout.strip()
print("\nthe pipeline the run used was committed as", PIPELINE_COMMIT[:10])


Path          accuracy    f1      n_test    n_train    roc_auc
metrics.json  0.8099      0.8129  1247      3753       0.8874

the pipeline the run used was committed as 4e2c17fd93


---
# Part 3 — What the run recorded   ·   session 7

The training script logged params, metrics and — the part that matters here — three tags:
which **data** (`dvc_data_md5`), which **code** (`git_commit`), and whether the tree was
**clean** when it ran (`git_dirty`). The third one decides whether the second one means anything.


In [13]:
import mlflow, json
from pathlib import Path

mlflow.set_tracking_uri(f"sqlite:///{Path.cwd()}/mlflow.db")
RUN_ID = Path("run_id.txt").read_text().strip()
run = mlflow.get_run(RUN_ID)

print("run id:", RUN_ID)
print("\nparams :", {k: v for k, v in run.data.params.items()})
print("metrics:", {k: round(v, 4) for k, v in run.data.metrics.items()})
print("\nthe two tags that matter:")
for k in ("dvc_data_md5", "git_commit", "splitter"):
    print(f"  {k:14} = {run.data.tags[k]}")
assert run.data.tags["dvc_data_md5"] == DATA_MD5

# the honesty check: a run logged from a dirty tree is not reproducible, whatever its tags say
print(f"\ngit_dirty is {run.data.tags['git_dirty']} -- so git_commit really describes this run")
assert run.data.tags["git_dirty"] == "false", "a dirty tree makes git_commit meaningless"
assert run.data.tags["git_commit"] == PIPELINE_COMMIT

2026/08/24 21:29:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/24 21:29:44 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


run id: e1d76959c7a04cf7b3987f8270044bc8

params : {'test_size': '0.25', 'random_state': '42', 'n_estimators': '200', 'max_depth': '8'}
metrics: {'roc_auc': 0.8874, 'accuracy': 0.8099, 'f1': 0.8129}

the two tags that matter:
  dvc_data_md5   = 977ddb47130abcf45cf9b16e2ccc74f7
  git_commit     = 4e2c17fd93bbf9d84679f93500b3aadd6186eb7e
  splitter       = GroupShuffleSplit(customer_id)

git_dirty is false -- so git_commit really describes this run


---
# Part 4 — Serve it   ·   sessions 3 and 10

The API loads the artifact the pipeline produced, validates input with Pydantic, reports which
model version and run answered, and exposes `/metrics` for Prometheus.

In [14]:
%%writefile app/main.py
"""The serving layer: sessions 3 (validation) and 10 (instrumentation) together."""
import os
import time
from typing import Literal

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException, Response
from prometheus_client import CONTENT_TYPE_LATEST, Counter, Gauge, Histogram, generate_latest
from pydantic import BaseModel, ConfigDict, Field

ARTIFACT = os.getenv("MODEL_PATH", "models/model.joblib")
STATE = {}

app = FastAPI(title="Churn API (end to end)", version="1.0.0")

REQUESTS = Counter("api_requests_total", "requests", ["endpoint", "status"])
LATENCY = Histogram("api_request_duration_seconds", "duration", ["endpoint"],
                    buckets=(.001, .005, .01, .025, .05, .1, .25, .5))
SCORE = Histogram("model_score", "predicted probability",
                  buckets=tuple(i / 10 for i in range(11)))
MODEL_INFO = Gauge("model_info", "1, labelled with provenance", ["version", "run_id"])


class Customer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tenure_months:  int   = Field(ge=0, le=600)
    monthly_charge: float = Field(ge=0, le=10_000)
    support_calls:  int   = Field(ge=0, le=100)
    plan: Literal["basic", "plus", "pro"]


@app.on_event("startup")
def load():
    b = joblib.load(ARTIFACT)
    STATE["b"] = b
    MODEL_INFO.labels(version=b["version"], run_id=b["run_id"][:8]).set(1)


@app.middleware("http")
async def observe(request, call_next):
    t0 = time.perf_counter()
    response = await call_next(request)
    LATENCY.labels(endpoint=request.url.path).observe(time.perf_counter() - t0)
    REQUESTS.labels(endpoint=request.url.path, status=str(response.status_code)).inc()
    return response


@app.get("/health")
def health():
    b = STATE.get("b")
    return {"status": "ok" if b else "degraded", "model_loaded": b is not None,
            "version": b["version"] if b else None,
            "run_id": b["run_id"][:8] if b else None,
            "offline_roc_auc": b["metrics"]["roc_auc"] if b else None}


@app.post("/predict")
def predict(c: Customer):
    b = STATE.get("b")
    if b is None:
        raise HTTPException(503, "model not loaded")
    frame = pd.DataFrame([c.model_dump()])[b["features"]]
    p = float(b["pipeline"].predict_proba(frame)[0, 1])
    SCORE.observe(p)
    return {"churn": p >= .5, "probability": round(p, 4),
            "version": b["version"], "run_id": b["run_id"][:8]}


@app.get("/metrics")
def metrics():
    return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)

Writing app/main.py


`TestClient` runs the app in-process — no server, no port, no waiting. It is the fastest way to
check an API, and it is what you would run in CI.

In [15]:
import sys
sys.path.insert(0, ".")
from fastapi.testclient import TestClient
from app.main import app

with TestClient(app) as c:
    h = c.get("/health").json()
    print("GET /health:", h)
    assert h["model_loaded"] and h["run_id"] == RUN_ID[:8]
    print(f"  the API is serving run {h['run_id']}, offline roc_auc {h['offline_roc_auc']}")

    r = c.post("/predict", json={"tenure_months": 2, "monthly_charge": 130.0,
                                 "support_calls": 8, "plan": "basic"})
    print("POST /predict:", r.status_code, r.json())

    bad = c.post("/predict", json={"tenure_months": -2, "plan": "gold"})
    print("bad request ->", bad.status_code)

    metrics_text = c.get("/metrics").text
info = [l for l in metrics_text.splitlines() if l.startswith("model_info")]
print("\nprovenance is exposed as a metric, so Prometheus can alert on the wrong model:")
print(" ", info[0] if info else "(none)")
assert RUN_ID[:8] in metrics_text

GET /health: {'status': 'ok', 'model_loaded': True, 'version': '1.0.0', 'run_id': 'e1d76959', 'offline_roc_auc': 0.8874}
  the API is serving run e1d76959, offline roc_auc 0.8874
POST /predict: 200 {'churn': True, 'probability': 0.9996, 'version': '1.0.0', 'run_id': 'e1d76959'}
bad request -> 422

provenance is exposed as a metric, so Prometheus can alert on the wrong model:
  model_info{run_id="e1d76959",version="1.0.0"} 1.0


---
# Part 5 — Ship it   ·   session 4

One image, multi-stage, non-root, with the model mounted rather than baked in — so a new model
does not require a new image.

In [16]:
%%writefile requirements.txt
fastapi==0.115.2
uvicorn==0.31.1
pydantic==2.11.10
scikit-learn==1.6.1
pandas==2.2.3
joblib==1.4.2
prometheus-client==0.21.0

Writing requirements.txt


The `Dockerfile` is the recipe for the image. It is **multi-stage**: the first stage installs the
dependencies, the second copies only the installed files across, so the compilers never ship.

In [17]:
%%writefile Dockerfile
FROM python:3.12-slim AS builder
WORKDIR /srv
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

FROM python:3.12-slim
RUN useradd --create-home --uid 10001 appuser
WORKDIR /srv

COPY --from=builder /install /usr/local
COPY app/ ./app/

USER appuser
ENV MODEL_PATH=/srv/models/model.joblib PYTHONUNBUFFERED=1
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


`.dockerignore` keeps things out of the build. Without it, `docker build` would send the whole
folder — including the data and the git history — to the Docker daemon.

In [18]:
%%writefile .dockerignore
.git
.dvc/cache
__pycache__/
*.ipynb
mlruns/
mlflow.db
data/
models/

Writing .dockerignore


Now build the image. This takes a minute the first time and seconds afterwards, because Docker
caches each step.

In [19]:
import subprocess, time
r = subprocess.run(["docker", "build", "-t", "e2e-churn:1.0.0", "."],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "the image must build"
size = int(subprocess.run(["docker", "image", "inspect", "-f", "{{.Size}}", "e2e-churn:1.0.0"],
                          capture_output=True, text=True).stdout.strip())
# No build time reported on purpose: this cache is warm from earlier runs, and a warm
# build time says nothing. Session 4 measures cold builds, which is the honest way.
print(f"built e2e-churn:1.0.0 -- {size/1024/1024:.0f} MB")

built e2e-churn:1.0.0 -- 451 MB


Start the container and talk to it over HTTP. `-v` **mounts** your `models/` folder inside the
container, so the image does not carry the model — a new model is a new file, not a new image.

In [20]:
import socket, urllib.request, json as _json

def free_port(start=8500):
    for p in range(start, start + 200):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", p)) != 0:
                return p

PORT = free_port()
subprocess.run(["docker", "rm", "-f", "e2e-churn"], capture_output=True)
run = subprocess.run(
    ["docker", "run", "-d", "--name", "e2e-churn", "-p", f"{PORT}:8000",
     "-v", f"{os.getcwd()}/models:/srv/models:ro",     # the model, mounted not baked
     "--memory", "1g", "--cpus", "1.5", "e2e-churn:1.0.0"],
    capture_output=True, text=True)
print("container:", (run.stdout or run.stderr).strip()[:12])

for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1); break
    except Exception: time.sleep(1)

import requests
print("\nGET /health :", requests.get(f"http://127.0.0.1:{PORT}/health").json())
print("POST /predict:", requests.post(f"http://127.0.0.1:{PORT}/predict",
      json={"tenure_months": 2, "monthly_charge": 130.0,
            "support_calls": 8, "plan": "basic"}).json())
print("\nA container is serving the model that the versioned pipeline produced.")

container: be49cae3e096

GET /health : {'status': 'ok', 'model_loaded': True, 'version': '1.0.0', 'run_id': 'e1d76959', 'offline_roc_auc': 0.8874}
POST /predict: {'churn': True, 'probability': 0.9996, 'version': '1.0.0', 'run_id': 'e1d76959'}

A container is serving the model that the versioned pipeline produced.


---
# Part 6 — The proof   ·   all of them

Here is the claim from the top of the notebook. We have a run id and nothing else. From it:

1. read the **git commit** and the **data fingerprint** off the MLflow run
2. `git checkout` that commit and `dvc checkout` that data
3. `dvc repro --force` — run the pipeline again from scratch
4. compare the metric

If step 4 matches, the whole chain held.

In [21]:
# pretend it is six months later and all we have is this string
mystery_run = RUN_ID
print("all we know:", mystery_run)

r = mlflow.get_run(mystery_run)
want_commit = r.data.tags["git_commit"]
want_data   = r.data.tags["dvc_data_md5"]
want_auc    = r.data.metrics["roc_auc"]
print(f"\nthe run says:")
print(f"  code : {want_commit[:12]}")
print(f"  data : {want_data}")
print(f"  score: roc_auc {want_auc:.4f}")

all we know: e1d76959c7a04cf7b3987f8270044bc8

the run says:
  code : 4e2c17fd93bb
  data : 977ddb47130abcf45cf9b16e2ccc74f7
  score: roc_auc 0.8874


Now break things on purpose. More rows, fewer and shallower trees: a naive re-run cannot land on
the old score any more. This is what makes the next cell a real test rather than a coincidence.

In [22]:
# change the world, so a naive re-run would NOT reproduce anything
df2 = pd.concat([df, df.sample(1500, random_state=9)], ignore_index=True)
df2.to_csv("data/raw.csv", index=False)
!dvc add data/raw.csv -q
cfg = (open("params.yaml").read().replace("n_estimators: 200", "n_estimators: 40")
                                 .replace("max_depth: 8", "max_depth: 2"))
open("params.yaml", "w").write(cfg)
!git add -A && git commit -q -m "exp: more rows, fewer and shallower trees"
!dvc repro -q
drifted = json.load(open("metrics.json"))
print(f"after changing data AND params: roc_auc {drifted['roc_auc']:.4f}  "
      f"(the recorded run was {want_auc:.4f})")

⠋ Checking graph
/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2026/08/24 21:29:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/24 21:29:50 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
metrics: {'roc_auc': 0.8594, 'accuracy': 0.7552, 'f1': 0.7627, 'n_train': 

Travel back using **only** what the run recorded. `git checkout` restores the code; `dvc checkout`
restores the data that matches the pointers in that commit.

In [23]:
# now travel back using ONLY what the run recorded
import subprocess
subprocess.run(["git", "stash", "-q", "-u"], capture_output=True)
r = subprocess.run(["git", "checkout", "-q", want_commit], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
subprocess.run(["dvc", "checkout", "-q"], capture_output=True)

recovered_md5 = yaml.safe_load(open("data/raw.csv.dvc"))["outs"][0]["md5"]
print("data fingerprint now:", recovered_md5)
print("run wanted          :", want_data)
assert recovered_md5 == want_data, "dvc checkout must restore exactly the recorded data"
print("\nthe data matches the fingerprint the run recorded.")

data fingerprint now: 977ddb47130abcf45cf9b16e2ccc74f7
run wanted          : 977ddb47130abcf45cf9b16e2ccc74f7

the data matches the fingerprint the run recorded.


Run the pipeline again at that commit. If everything was recorded properly, the score comes back
identical — and the `assert` fails loudly if it does not.

In [24]:
# never hide stderr from a step you are about to assert on
r = subprocess.run(["dvc", "repro", "--force"], capture_output=True, text=True)
print((r.stdout or "")[-600:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-1500:])
assert r.returncode == 0, "the reproduction run must succeed"

reproduced = json.load(open("metrics.json"))
print(f"recorded   roc_auc: {want_auc:.4f}")
print(f"reproduced roc_auc: {reproduced['roc_auc']:.4f}")
print(f"difference        : {abs(reproduced['roc_auc'] - want_auc):.6f}")
assert abs(reproduced["roc_auc"] - want_auc) < 1e-9, \
    "a run identified only by its id must reproduce exactly"
print("\nIdentical. From one run id: the code, the data, and the same number back.")


Verifying data sources in stage: 'data/raw.csv.dvc'

Running stage 'train':
> python src/train.py
metrics: {'roc_auc': 0.8874, 'accuracy': 0.8099, 'f1': 0.8129, 'n_train': 3753, 'n_test': 1247}
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock .gitignore data/raw.csv.dvc models/.gitignore

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.

recorded   roc_auc: 0.8874
reproduced roc_auc: 0.8874
difference        : 0.000000

Identical. From one run id: the code, the data, and the same number back.


Back to the tip of the branch. `-f` throws away what the reproduction left behind.

In [25]:
# and back to where we were
!git checkout -q -f main && dvc checkout -q
print("returned to main")

returned to main


---
# Part 7 — What each session contributed, and what is missing

| Session | Contribution to the chain above |
|---|---|
| 1 · leakage | the pipeline, and `GroupShuffleSplit` because customers repeat |
| 3 · API | Pydantic validation, `/health`, the response naming its own version |
| 4 · Docker | one artifact, non-root, model mounted rather than baked |
| 6 · DVC | the data fingerprint, and `dvc repro` as the recipe |
| 7 · MLflow | the run record, and the two tags that made Part 6 possible |
| 10 · monitoring | `/metrics`, and `model_info` exposing provenance |
| 11 · automation | the workflow that would run all of this on a schedule, with a gate |

## What this notebook does **not** prove

Being honest about the boundary matters more than a tidy diagram:

* **No cluster.** Sessions 8 and 11 ran locally. Real distribution introduces failures — partial
  results, stragglers, network partitions — that one machine never shows you.
* **No accuracy monitoring.** Session 10 watches the *prediction distribution*, because labels
  arrive weeks later. Closing that loop needs a feedback pipeline this course does not build.
* **No feature store here.** Session 9 covered it; this chain reads a CSV. With several models
  sharing features you would put session 9 between Parts 1 and 2.
* **No real infrastructure.** Session 12 used the Docker provider. A cloud account brings IAM,
  networking and cost that change the design.
* **One team, one repo.** The hard parts of MLOps at scale are organisational: who owns the
  feature, who is paged, who may promote a model.

## The order to adopt this at work

Not all at once, and not in the order the diagram suggests:

1. **Make the metric honest** (session 1). Everything downstream optimises whatever you measure.
2. **Version the data** (session 6). Without it nothing else is reproducible.
3. **Record the runs** (session 7), with the two tags. Cheap, and it makes Part 6 possible.
4. **Serve it properly** (sessions 3, 4). Now other people can use it.
5. **Watch it** (session 10). One alert beats ten dashboards.
6. **Automate, with a gate** (session 11). Last, because automation multiplies whatever you had.

## Cleanup

In [26]:
import subprocess, shutil, os
subprocess.run(["docker", "rm", "-f", "e2e-churn"], capture_output=True)
subprocess.run(["docker", "rmi", "-f", "e2e-churn:1.0.0"], capture_output=True)
print("container and image removed")

os.chdir(BASE)
shutil.rmtree(PROJ, ignore_errors=True)
shutil.rmtree(BASE / "e2e_dvc_remote", ignore_errors=True)
print("sandbox removed")

container and image removed
sandbox removed
